<a href="https://colab.research.google.com/github/feliperangel204-cpu/proyecto-aplicado-1/blob/main/notebooks/Proyecto-aplicado-Desa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Proyecto aplicado 1

Luis Felipe Rangel
Daniel Alarcón
Edward Atehortua
Angie Tamara
Frank Mendoza


# Clasificación automática de etiquetas industriales para la prevención de crosspacking

**Proyecto aplicado · Maestría en Ciencia de Datos · Universidad Icesi**
**Empresa aliada: MV Perception**

Luis Felipe Rangel · Daniel Alarcón · Edward Atehortua · Angie Tamara · Frank Mendoza

---

## El problema

En la línea de envasado se produce *crosspacking*: un producto sale con la etiqueta de una
referencia distinta a la que indica la orden de producción. La línea procesa 250 unidades por
minuto, es decir una cada 240 milisegundos, alterna 43 referencias y opera de forma continua.

Este cuaderno desarrolla y evalúa un modelo de clasificación de imágenes que identifica la
referencia a partir de su etiqueta, para contrastarla con la orden de producción.

## Las metas que fija la empresa

| Meta | Valor | Cómo se verifica en este cuaderno |
|---|---|---|
| Precisión | superior al 95 % | Sección 4, con el protocolo de partición declarado en la sección 2 |
| Tiempo de inferencia | inferior a 240 ms | Sección 5, percentil 95 sobre la cadena completa |

## Estructura del cuaderno

| Sección | Contenido |
|---|---|
| 0 | Configuración: librerías, conexión con Drive y parámetros |
| 1 | Comprensión de los datos: inventario, integridad, duplicados, procedencia y distribución |
| 2 | Preparación: depuración, protocolo de partición y preprocesamiento |
| 3 | Modelado: línea base clásica, red propia y tres arquitecturas con transferencia |
| 4 | Evaluación: métricas globales, por clase y sobre los pares difíciles |
| 5 | Latencia: medición de la cadena completa contra el presupuesto de la línea |
| 6 | Comparativo entre modelos y conclusiones |

## Cómo reproducirlo

Las imágenes no están en el repositorio: pesan unos 10 GB y viven en Google Drive.
La sección 0 monta Drive y verifica que el conjunto esté completo antes de ejecutar nada más.

Las operaciones costosas guardan su resultado en caché dentro de `pa1-analisis`, de modo que
una interrupción de Colab no obliga a repetirlas. Las particiones y las semillas quedan fijadas
en la sección 0, así que dos ejecuciones distintas producen los mismos subconjuntos y los
mismos resultados.

**Nota sobre la partición.** Las imágenes de una misma secuencia de disparos no son
observaciones independientes. Repartirlas al azar entre entrenamiento y prueba infla el
resultado: la sección 1 lo mide sobre este mismo conjunto. El protocolo de partición que usa
este cuaderno se define y se justifica en la sección 2.

## 0. Configuración

Librerías, conexión con Google Drive y parámetros del análisis. Las rutas y los umbrales
quedan reunidos en una sola celda: si cambia la ubicación del conjunto o se ajusta un criterio,
se modifica ahí y no en el resto del cuaderno.

In [1]:
import os
import re
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 60)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

print('numpy', np.__version__, '· pandas', pd.__version__, '· Pillow', Image.__version__)

numpy 2.1.3 · pandas 2.2.3 · Pillow 11.3.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ---------- Rutas. Ajustar si las carpetas tienen otro nombre en Drive ----------
RAIZ   = Path('/content/drive/MyDrive/pa1-imagenes')    # contiene las 43 carpetas de referencias
SALIDA = Path('/content/drive/MyDrive/pa1-analisis')    # caché, tablas y figuras
SALIDA.mkdir(parents=True, exist_ok=True)

# ---------- Reproducibilidad ----------
SEMILLA = 42
np.random.seed(SEMILLA)
rng = np.random.default_rng(SEMILLA)

# ---------- Umbrales del análisis exploratorio ----------
CORTE_SESION = 600     # segundos sin disparos que definen otra jornada de trabajo
CORTE_RAFAGA = 3       # segundos sin disparos que definen otra ráfaga
RMSE_CASI    = 4.0     # error cuadrático medio bajo el cual dos imágenes son casi la misma
LADO_MINI    = 256     # lado largo de la miniatura usada para comparar imágenes

# ---------- Parámetros de la partición ----------
FRAC_PRUEBA = 0.30     # proporción de ráfagas que van a prueba en las clases de una sola jornada
FRAC_VALID  = 0.20     # proporción de ráfagas de entrenamiento que van a validación

# ---------- Características del montaje, usadas para interpretar resoluciones ----------
ANCHO_NATIVO = 1440
ALTO_NATIVO  = 944
MM_POR_PIXEL = 240 / ANCHO_NATIVO   # campo de visión horizontal de 240 mm

print(f'origen : {RAIZ}')
print(f'salida : {SALIDA}')
print(f'escala : {MM_POR_PIXEL:.4f} mm por píxel')

origen : /content/drive/MyDrive/pa1-imagenes
salida : /content/drive/MyDrive/pa1-analisis
escala : 0.1667 mm por píxel


In [10]:
assert RAIZ.exists(), f'No existe la ruta {RAIZ}. Revisa el nombre de la carpeta en Drive.'

carpetas = sorted(
    [d for d in RAIZ.iterdir() if d.is_dir() and not d.name.startswith(('_', '.'))],
    key=lambda d: int(d.name.split('_')[0]) if d.name.split('_')[0].isdigit() else 999
)

conteo = {c.name: len([f for f in os.listdir(c) if f.lower().endswith('.bmp')]) for c in carpetas}
total = sum(conteo.values())

print(f'{len(carpetas)} carpetas · {total} imágenes BMP\n')
for nombre, n in conteo.items():
    print(f'  {nombre:36s} {n:>5}')

ESPERADAS_CARPETAS = 43
ESPERADAS_IMAGENES = 2605
if len(carpetas) != ESPERADAS_CARPETAS or total != ESPERADAS_IMAGENES:
    print(f'\nAVISO: se esperaban {ESPERADAS_CARPETAS} carpetas y {ESPERADAS_IMAGENES} imágenes.')
    print('Si la subida a Drive todavía no termina, conviene esperar antes de continuar.')
else:
    print('\nEl conjunto está completo.')

43 carpetas · 2605 imágenes BMP

  0_MULTIVIAMINAS_SAVITAL_510             37
  1_ANTICASPA_SAVITAL_510                 74
  2_COLAGENO_SAVITAL_510                  64
  3_FUSION_SAVITAL_490                    41
  4_FUSION_SAVITAL_510                   107
  5_HIALURONICO_SAVITAL_510               31
  6_MULTIOLEOS_SAVITAL_510                24
  7_SERUM_SAVITAL_490                     51
  8_ARGAN_SAVITAL_510                     41
  9_ANTICASPA_LIMPIEZA_SAVITAL_510       133
  10_COLAGENO_SAVITAL_510                 35
  11_SERUM_SAVITAL_490                    47
  12_ELIXIR_SAVITAL_510                   19
  13_DOVE_NUTRICION_370                  135
  14_DOVE_RECONSTRUCCION_370              49
  15_DOVE_REGENERACION_370                92
  16_DOVE_REPAIR_350                      87
  17_DOVE_HIDRA_SUAVE_370                111
  18_DOVE_HIDRA_370                       24
  19_SEDAL_UV_340                         52
  20_SEDAL_ZERO_CASPA_340                 95
  21_SEDAL_RIZOS_DEFIN

## 1. Comprensión de los datos

Esta sección reconstruye el inventario del conjunto desde los archivos, verifica su
integridad, identifica duplicados, reconstruye la procedencia temporal de cada imagen y
mide el desbalance entre clases.

Cierra con la medida que más condiciona el resto del cuaderno: cuánto cambia el resultado
según cómo se reparta el conjunto entre entrenamiento y prueba.

### 1.1 Inventario

Una sola pasada sobre las 2.607 imágenes. En ese recorrido se calcula, por archivo:

- el **resumen MD5** del contenido, que identifica archivos idénticos byte a byte;
- la **cabecera del BMP**, que da resolución, modo de color y número de canales;
- la **marca de tiempo** codificada en el nombre, que da la procedencia temporal;
- las **estadísticas fotométricas** por canal, que miden la variación de iluminación;
- una **miniatura en gris**, que permite comparar imágenes entre sí sin volver a leer los 10 GB.

Hacerlo todo en un solo recorrido evita leer el conjunto cinco veces. El resultado queda en
caché, de modo que una desconexión de Colab no obliga a repetir la pasada.

In [14]:
# ── 1.1 Inventario: una sola pasada por el dataset ──
# Lee cada BMP una vez y de esa lectura saca todo lo que se necesita después:
# huella MD5, encabezado, marca de tiempo del nombre, estadística fotométrica
# y una miniatura en gris. Guarda avance por carpeta para poder retomar.
import struct, hashlib
from io import BytesIO

SALIDA = Path('/content/drive/MyDrive/pa1-salidas')   # resultados, fuera del dataset
SALIDA.mkdir(parents=True, exist_ok=True)
assert RAIZ.is_dir(),  f'No existe el dataset en {RAIZ}'
assert SALIDA != RAIZ, 'Las salidas no pueden ir dentro del dataset'
print('Dataset:', RAIZ, '\nSalidas:', SALIDA)

PATRON    = re.compile(r'^Imagen?_(\d+)_(\d{14})', re.IGNORECASE)
RUTA_INV  = SALIDA / 'inventario.csv'
RUTA_MINI = SALIDA / 'miniaturas.npz'
ALTO_MINI = round(LADO_MINI * ALTO_NATIVO / ANCHO_NATIVO)

COLUMNAS = ['carpeta','archivo','ruta','bytes','md5','ancho','alto','alto_crudo',
            'bpp','canales','compresion','offset_datos','tam_cabecera',
            'indice','marca_tiempo','media','desv','p01','p50','p99',
            'frac_oscuros','frac_quemados']

def leer_cabecera(b):
    """Campos del BITMAPFILEHEADER + BITMAPINFOHEADER, o None si no es BMP."""
    if b[:2] != b'BM':
        return None
    offset,  = struct.unpack('<I', b[10:14])
    tam_cab, = struct.unpack('<I', b[14:18])
    ancho, alto = struct.unpack('<ii', b[18:26])
    _planos, bpp = struct.unpack('<HH', b[26:30])
    compresion, = struct.unpack('<I', b[30:34])
    return offset, tam_cab, ancho, alto, bpp, compresion

def medir(ruta_abs, carpeta, archivo):
    datos = ruta_abs.read_bytes()
    cab = leer_cabecera(datos[:54])
    if cab is None:
        offset = tam_cab = ancho = alto = bpp = compresion = -1
    else:
        offset, tam_cab, ancho, alto, bpp, compresion = cab
    g = np.asarray(Image.open(BytesIO(datos)).convert('L'))
    mini = np.asarray(Image.fromarray(g).resize((LADO_MINI, ALTO_MINI)), dtype=np.uint8)
    m = PATRON.match(archivo)
    p01, p50, p99 = np.percentile(g, [1, 50, 99])
    fila = dict(
        carpeta=carpeta, archivo=archivo, ruta=f'{carpeta}/{archivo}',
        bytes=len(datos), md5=hashlib.md5(datos).hexdigest(),
        ancho=ancho, alto=abs(alto), alto_crudo=alto, bpp=bpp,
        canales=max(bpp // 8, 1), compresion=compresion,
        offset_datos=offset, tam_cabecera=tam_cab,
        indice=int(m.group(1)) if m else -1,
        marca_tiempo=m.group(2) if m else '',
        media=round(float(g.mean()), 3), desv=round(float(g.std()), 3),
        p01=float(p01), p50=float(p50), p99=float(p99),
        frac_oscuros=round(float((g <= 5).mean()), 5),
        frac_quemados=round(float((g >= 250).mean()), 5))
    return fila, mini

# --- lo que ya esté calculado se reutiliza ---
hecho_filas, hecho_mini = {}, {}
if RUTA_INV.exists():
    prev = pd.read_csv(RUTA_INV, dtype={'marca_tiempo': str})
    if set(COLUMNAS).issubset(prev.columns):
        prev = prev.drop_duplicates('ruta')
        hecho_filas = {r['ruta']: r for r in prev[COLUMNAS].to_dict('records')}
        if RUTA_MINI.exists():
            z = np.load(RUTA_MINI, allow_pickle=False)
            hecho_mini = dict(zip(z['rutas'].tolist(), z['mini']))
        print(f'Caché: {len(hecho_filas)} filas y {len(hecho_mini)} miniaturas reutilizables.')
    else:
        print('La caché existente tiene otras columnas. Se recalcula desde cero.')
else:
    print('Sin caché previa. Primera pasada completa.')

carpetas = sorted([d for d in RAIZ.iterdir() if d.is_dir()], key=lambda d: d.name)
filas, minis, claves = [], [], []
t0, nuevas = time.time(), 0

for i, carp in enumerate(carpetas, 1):
    archivos = sorted(p.name for p in carp.iterdir() if p.suffix.lower() == '.bmp')
    for nombre in archivos:
        clave = f'{carp.name}/{nombre}'
        if clave in hecho_filas and clave in hecho_mini:
            fila, mini = hecho_filas[clave], hecho_mini[clave]
        else:
            fila, mini = medir(carp / nombre, carp.name, nombre)
            nuevas += 1
        filas.append(fila); minis.append(mini); claves.append(clave)
    try:
        pd.DataFrame(filas)[COLUMNAS].to_csv(RUTA_INV, index=False)
        if i % 10 == 0 or i == len(carpetas):
            np.savez_compressed(RUTA_MINI, mini=np.stack(minis), rutas=np.array(claves))
    except OSError as e:
        print('  aviso: no se guardó el avance de esta carpeta →', e)
    print(f'[{i:2d}/{len(carpetas)}] {carp.name:40s} {len(archivos):4d} img · '
          f'leídas de Drive {nuevas:4d} · {(time.time()-t0)/60:5.1f} min')

inv = pd.DataFrame(filas)[COLUMNAS]
inv['marca_tiempo'] = pd.to_datetime(inv.marca_tiempo, format='%Y%m%d%H%M%S', errors='coerce')
MINI, CLAVES_MINI = np.stack(minis), claves

print(f'\nInventario: {len(inv)} filas · {inv.carpeta.nunique()} carpetas · '
      f'{inv.bytes.sum()/2**30:.2f} GiB · miniaturas {MINI.shape}')
print('Coincide con lo esperado:', len(inv) == ESPERADAS_IMAGENES
      and inv.carpeta.nunique() == ESPERADAS_CARPETAS)

Dataset: /content/drive/MyDrive/pa1-imagenes 
Salidas: /content/drive/MyDrive/pa1-salidas
Sin caché previa. Primera pasada completa.
[ 1/43] 0_MULTIVIAMINAS_SAVITAL_510                37 img · leídas de Drive   37 ·   0.0 min
[ 2/43] 10_COLAGENO_SAVITAL_510                    35 img · leídas de Drive   72 ·   0.1 min
[ 3/43] 11_SERUM_SAVITAL_490                       47 img · leídas de Drive  119 ·   0.2 min
[ 4/43] 12_ELIXIR_SAVITAL_510                      19 img · leídas de Drive  138 ·   0.2 min
[ 5/43] 13_DOVE_NUTRICION_370                     135 img · leídas de Drive  273 ·   0.3 min
[ 6/43] 14_DOVE_RECONSTRUCCION_370                 49 img · leídas de Drive  322 ·   0.4 min
[ 7/43] 15_DOVE_REGENERACION_370                   92 img · leídas de Drive  414 ·   0.5 min
[ 8/43] 16_DOVE_REPAIR_350                         87 img · leídas de Drive  501 ·   1.0 min
[ 9/43] 17_DOVE_HIDRA_SUAVE_370                   111 img · leídas de Drive  612 ·   1.3 min
[10/43] 18_DOVE_HIDRA_370     

### 1.2 Integridad del formato

El BMP que escribe la cámara no lleva metadatos: el encabezado ocupa 54 bytes
y a partir de ahí todo es píxel. La procedencia de cada imagen se reconstruye
entonces desde dos fuentes, el encabezado y el nombre del archivo
(`Image_NNNN_AAAAMMDDhhmmss`), y conviene confirmar que ambas son confiables
antes de derivar cualquier cifra de ellas. Tres comprobaciones:
que la geometría declarada sea la misma en todo el conjunto, que el tamaño en
disco coincida con la aritmética `54 + ancho × alto × canales` —un archivo
truncado por una carga interrumpida falla aquí—, y que cada nombre entregue
índice y marca de tiempo legibles.

In [15]:
# ── 1.2 Integridad del formato ──
geom = (inv.groupby(['ancho','alto','alto_crudo','bpp','canales','compresion','offset_datos'])
           .size().rename('imagenes').reset_index())
print('Geometrías declaradas en el encabezado:')
display(geom)

inv['bytes_esperados'] = 54 + inv.ancho * inv.alto * inv.canales
desajuste = inv[inv.bytes != inv.bytes_esperados]
print(f'\nArchivos cuyo tamaño no coincide con el encabezado: {len(desajuste)}')
if len(desajuste):
    display(desajuste[['ruta','bytes','bytes_esperados','ancho','alto','canales']].head(20))

no_bmp = inv[inv.ancho < 0]
print(f'Archivos que no son BMP legibles: {len(no_bmp)}')

sin_patron = inv[inv.indice < 0]
print(f'Nombres fuera del patrón Image_NNNN_AAAAMMDDhhmmss: {len(sin_patron)}')
if len(sin_patron):
    display(sin_patron[['carpeta','archivo']].head(20))

print(f'Marcas de tiempo ilegibles: {int(inv.marca_tiempo.isna().sum())}')
print(f'Rango temporal: {inv.marca_tiempo.min()}  →  {inv.marca_tiempo.max()}')

repetidos = inv[inv.duplicated('ruta', keep=False)]
print(f'\nNombres repetidos dentro de una misma carpeta: {len(repetidos)}')
if len(repetidos):
    display(repetidos[['carpeta','archivo','bytes','md5']].sort_values(['carpeta','archivo']))

print(f'\nResolución única en todo el conjunto: '
      f'{inv.ancho.nunique() == 1 and inv.alto.nunique() == 1}')
print(f'Escala espacial: {MM_POR_PIXEL:.4f} mm por píxel '
      f'(FOV 240 mm / {ANCHO_NATIVO} px)')

Geometrías declaradas en el encabezado:


,ancho,alto,alto_crudo,bpp,canales,compresion,offset_datos,imagenes
0,944,1440,1440,24,3,0,54,1
1,1440,944,944,24,3,0,54,2599
2,1440,1080,1080,24,3,0,54,5



Archivos cuyo tamaño no coincide con el encabezado: 0
Archivos que no son BMP legibles: 0
Nombres fuera del patrón Image_NNNN_AAAAMMDDhhmmss: 1


,carpeta,archivo
1387,29_SEDAL_ZERO_CASPA_650,123.bmp


Marcas de tiempo ilegibles: 1
Rango temporal: 2024-12-28 16:28:32  →  2025-07-24 15:30:25

Nombres repetidos dentro de una misma carpeta: 0

Resolución única en todo el conjunto: False
Escala espacial: 0.1667 mm por píxel (FOV 240 mm / 1440 px)
